# 01 · Ingesta — Capology

Notebook de scraping de datos salariales de jugadores desde **Capology** usando la librería `ScraperFC`.

**Alcance:**
- 6 ligas: La Liga, Premier League, Serie A, Bundesliga, Ligue 1, Süper Lig
- Temporadas: 20/21, 21/22, 22/23, 23/24, 24/25, 25/26
- Todos los salarios convertidos a EUR

**Nota:** A diferencia de Sofascore, Capology publica los datos salariales de la temporada
en curso como definitivos desde el inicio, por lo que no es necesario un snapshot con fecha.

**Salida:** `data/raw/capology/cg_<liga>_<temporada>.csv`

---

In [1]:
import pandas as pd
import ScraperFC as sfc

In [ ]:
cg = sfc.Capology()

## 1. Exploración de ligas y temporadas disponibles

Se comprueba qué ligas y temporadas tiene disponibles Capology a través de ScraperFC,
para confirmar la cobertura antes de lanzar el scraping.

In [23]:
print("--- LIGAS DISPONIBLES EN CAPOLOGY ---")
for league_name in sfc.capology.comps.keys():
    print(league_name)

--- LIGAS DISPONIBLES EN CAPOLOGY ---
Belgium Pro League
England Premier League
England EFL Championship
France Ligue 1
France Ligue 2
Germany Bundesliga
Germany 2.Bundesliga
Italy Serie A
Italy Serie B
Netherlands Eredivisie
Portugal Primeira Liga
Scotland Premier League
Spain La Liga
Spain La Liga 2
Turkiye Super Lig


In [24]:
ligas_capology = [
    "Spain La Liga",
    "England Premier League",
    "Italy Serie A",
    "Germany Bundesliga",
    "France Ligue 1",
    "Turkiye Super Lig"
]

print("--- COMPROBANDO TEMPORADAS EN CAPOLOGY ---")

for liga in ligas_capology:
    try:
        # Capology devuelve directamente una lista, así que la usamos tal cual
        cg_seasons = cg.get_valid_seasons(league=liga)
        
        print(f"\n✅ {liga}:")
        print(f"   Temporadas: {cg_seasons[:6]}") # Mostramos las 6 más recientes
        
    except Exception as e:
        print(f"\n❌ Error para {liga}: {e}")

--- COMPROBANDO TEMPORADAS EN CAPOLOGY ---

✅ Spain La Liga:
   Temporadas: ['2025-26', '2024-25', '2023-24', '2022-23', '2021-22', '2020-21']

✅ England Premier League:
   Temporadas: ['2025-26', '2024-25', '2023-24', '2022-23', '2021-22', '2020-21']

✅ Italy Serie A:
   Temporadas: ['2025-26', '2024-25', '2023-24', '2022-23', '2021-22', '2020-21']

✅ Germany Bundesliga:
   Temporadas: ['2025-26', '2024-25', '2023-24', '2022-23', '2021-22', '2020-21']

✅ France Ligue 1:
   Temporadas: ['2025-26', '2024-25', '2023-24', '2022-23', '2021-22', '2020-21']

✅ Turkiye Super Lig:
   Temporadas: ['2025-26', '2024-25', '2023-24', '2022-23', '2021-22', '2020-21']


## 2. Scraping de salarios (20/21 → 25/26)

Se descargan los datos salariales de cada jugador por liga y temporada usando `scrape_salaries()`.
El parámetro `currency='eur'` normaliza todos los salarios a euros independientemente
de la liga (relevante especialmente para la Premier League, que publica en libras).
Se añaden las columnas `data_country` y `data_season` como metadatos de trazabilidad.

**SCRAPING TEMPORADAS 20/21,21/22,22/23,23/24,24/25,25/26**

In [37]:
# 1. Mapeo de las 6 ligas "temporada partida"
mapa_europa = {
    "Spain La Liga": "spain",
    "England Premier League": "england",
    "Italy Serie A": "italy",
    "Germany Bundesliga": "germany",
    "France Ligue 1": "france",
    "Turkiye Super Lig": "turkey"
}

# 2. Diccionario exacto: { Petición a Sofascore : Nombre final del archivo }
temporadas_europa = {
    '2020-21': '2021',
    '2021-22': '2122',
    '2022-23': '2223',
    '2023-24': '2324',
    '2024-25': '2425',
    '2025-26': '2526'
}

print("Iniciando descarga de Ligas Europeas...")

for liga_completa, pais_corto in mapa_europa.items():
    for req_year, sufijo_archivo in temporadas_europa.items():
        
        file_name = f"cg_{pais_corto}_{sufijo_archivo}.csv"
        
        try:
            print(f"📥 Descargando {liga_completa} ({req_year}) -> {file_name}...")
            
            df = cg.scrape_salaries(year=req_year, league=liga_completa, currency="eur")
            
            df['data_country'] = pais_corto
            df['data_season'] = sufijo_archivo
            
            df.to_csv(file_name, index=False)
            print(f"✅ Guardado: {file_name}")
            
        except Exception as e:
            print(f"❌ Error en {liga_completa} {req_year}: {e}")

Iniciando descarga de Ligas Europeas...
📥 Descargando Spain La Liga (2020-21) -> cg_spain_2021.csv...
Changed currency to eur.
✅ Guardado: cg_spain_2021.csv
📥 Descargando Spain La Liga (2021-22) -> cg_spain_2122.csv...
Changed currency to eur.
✅ Guardado: cg_spain_2122.csv
📥 Descargando Spain La Liga (2022-23) -> cg_spain_2223.csv...
Changed currency to eur.
✅ Guardado: cg_spain_2223.csv
📥 Descargando Spain La Liga (2023-24) -> cg_spain_2324.csv...
Changed currency to eur.
✅ Guardado: cg_spain_2324.csv
📥 Descargando Spain La Liga (2024-25) -> cg_spain_2425.csv...
Changed currency to eur.
✅ Guardado: cg_spain_2425.csv
📥 Descargando Spain La Liga (2025-26) -> cg_spain_2526.csv...
Changed currency to eur.
✅ Guardado: cg_spain_2526.csv
📥 Descargando England Premier League (2020-21) -> cg_england_2021.csv...
Changed currency to eur.
✅ Guardado: cg_england_2021.csv
📥 Descargando England Premier League (2021-22) -> cg_england_2122.csv...
Changed currency to eur.
✅ Guardado: cg_england_2122.cs